In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
import functools
import subprocess
import agents.mcp.server


agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

In [3]:
from backend.accounts import Account

In [4]:
account = Account.get("eric")
account.reset()
account

Account(name='eric', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [5]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "eric", "balance": 9065.91556, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 311.36148000000003, "timestamp": "2026-11-08 21:13:20", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-11-08 21:13:20", 9998.135559999999]], "total_portfolio_value": 9998.135559999999, "total_profit_loss": -1.8644400000011956}'

In [15]:
account.report()

'{"name": "eric", "balance": 9065.91556, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 311.36148000000003, "timestamp": "2026-11-08 21:13:20", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-11-08 21:13:20", 9998.135559999999], ["2026-11-08 21:13:29", 9998.10556], ["2026-11-08 21:27:34", 9996.24556]], "total_portfolio_value": 9996.24556, "total_profit_loss": -3.7544400000006135}'

In [16]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 311.36148000000003,
  'timestamp': '2026-11-08 21:13:20',
  'rationale': 'Because this bookstore website looks promising'}]

In [17]:
params = {"command": "uv", "args": ["run", "-m", "backend.accounts_server"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tool = await server.list_tools()

In [9]:
mcp_tool

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=

In [18]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is eric and my account is under the name eric. What's my balance and my holdings?"
model = "gpt-5.4-mini"

In [19]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Balance: 9065.91556  
Holdings: AMZN 3 shares